# 🚗 Car Detection using HOG + SVM + Sliding Window
**Dataset:** Car and Non-Car Dataset (Kaggle - lachlannegus)

**Pipeline:**
1. Prepare Dataset (Car / Non-Car)
2. Extract HOG Features
3. Train SVM Classifier
4. Sliding Window Detection
5. Draw Bounding Boxes
6. Evaluate Results

## 📦 Step 0: Install Libraries & Download Dataset

In [ ]:
!pip install kagglehub scikit-image scikit-learn matplotlib opencv-python-headless tqdm -q

In [ ]:
import kagglehub

# Download dataset from Kaggle
path = kagglehub.dataset_download("lachlannegus/car-and-non-car-dataset")
print("✅ Path to dataset files:", path)

In [ ]:
import os
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from skimage.feature import hog
from skimage import exposure
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported!')

## 📂 Step 1: Prepare the Dataset
بنفصل الصور لـ **Positive (cars)** و **Negative (non-cars)** ونعمل resize لـ 64×64

In [ ]:
# ─── Explore dataset structure ───────────────────────────────────────────────
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    n_files = len([f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if n_files > 0 or level < 3:
        print(f"{indent}📁 {folder_name}/  ({n_files} images)")

In [ ]:
# ─── Collect image paths ─────────────────────────────────────────────────────
# Adjust folder names if your dataset has different structure
exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG')

def collect_images(base_path, keyword):
    """Find all images in subdirectories matching keyword (case-insensitive)"""
    found = []
    for root, dirs, files in os.walk(base_path):
        if keyword.lower() in root.lower():
            for ext in exts:
                found += glob.glob(os.path.join(root, ext))
    return found

car_paths     = collect_images(path, 'car')
non_car_paths = collect_images(path, 'non')

# If the above doesn't work, list all subfolders and pick manually:
if len(car_paths) == 0 or len(non_car_paths) == 0:
    all_imgs = []
    for ext in exts:
        all_imgs += glob.glob(os.path.join(path, '**', ext), recursive=True)
    # Split by parent folder name
    car_paths     = [p for p in all_imgs if 'car'     in os.path.dirname(p).lower() and 'non' not in os.path.dirname(p).lower()]
    non_car_paths = [p for p in all_imgs if 'non'     in os.path.dirname(p).lower()]
    if len(non_car_paths) == 0:
        non_car_paths = [p for p in all_imgs if 'notcar'  in os.path.dirname(p).lower() or 'not_car' in os.path.dirname(p).lower()]

print(f'🚗 Car images     : {len(car_paths)}')
print(f'🌳 Non-car images : {len(non_car_paths)}')

assert len(car_paths) > 0 and len(non_car_paths) > 0, \
    "❌ Could not find images! Run the cell above to check folder names and update paths manually."

In [ ]:
# ─── Visualize Samples ───────────────────────────────────────────────────────
WIN_SIZE = (64, 64)  # fixed resize for all images

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
fig.suptitle('Dataset Samples  |  Positive: Cars     Negative: Non-Cars',
             fontsize=14, fontweight='bold')

for i in range(8):
    img = cv2.cvtColor(cv2.resize(cv2.imread(car_paths[i]), WIN_SIZE), cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img); axes[0, i].set_title('Car ✅', fontsize=9); axes[0, i].axis('off')

for i in range(8):
    img = cv2.cvtColor(cv2.resize(cv2.imread(non_car_paths[i]), WIN_SIZE), cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img); axes[1, i].set_title('Non-Car ❌', fontsize=9); axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('01_dataset_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 🔬 Step 2: Extract HOG Features

**HOG (Histogram of Oriented Gradients):** بيحسب اتجاه وقوة الـ gradients في كل cell صغيرة من الصورة.

| Parameter | Value | المعنى |
|---|---|---|
| `orientations` | 9 | عدد الـ bins (0°→180°) |
| `pixels_per_cell` | (8,8) | حجم كل cell |
| `cells_per_block` | (2,2) | normalization block |
| Color space | YCrCb | أفضل من RGB للـ HOG |
| Channels | ALL (3) | بنستخدم الـ 3 channels |

In [ ]:
# ─── HOG Parameters ──────────────────────────────────────────────────────────
ORIENTATIONS    = 9
PIX_PER_CELL    = (8, 8)
CELLS_PER_BLOCK = (2, 2)

def extract_hog_features(image_path):
    """
    Read image → resize to 64x64 → YCrCb → HOG on all 3 channels → concat
    Returns: 1D numpy feature vector
    """
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.resize(img, WIN_SIZE)
    img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)

    features = []
    for ch in range(3):
        feat = hog(
            img_ycrcb[:, :, ch],
            orientations=ORIENTATIONS,
            pixels_per_cell=PIX_PER_CELL,
            cells_per_block=CELLS_PER_BLOCK,
            transform_sqrt=True,
            feature_vector=True
        )
        features.append(feat)
    return np.concatenate(features)

feat_len = len(extract_hog_features(car_paths[0]))
print(f'📐 HOG feature vector length: {feat_len}')

In [ ]:
# ─── Visualize HOG on Car vs Non-Car ─────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('HOG Feature Visualization per Channel', fontsize=14, fontweight='bold')
ch_names = ['Y (Luma)', 'Cr', 'Cb']
samples  = [(car_paths[0], '🚗 Car'), (non_car_paths[0], '🌳 Non-Car')]

for row, (p, label) in enumerate(samples):
    img = cv2.imread(p)
    img_rgb   = cv2.cvtColor(cv2.resize(img, WIN_SIZE), cv2.COLOR_BGR2RGB)
    img_ycrcb = cv2.cvtColor(cv2.resize(img, WIN_SIZE), cv2.COLOR_BGR2YCrCb)

    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_title(f'{label}\nOriginal 64×64', fontsize=10)
    axes[row, 0].axis('off')

    for col in range(3):
        _, hog_img = hog(img_ycrcb[:, :, col], orientations=ORIENTATIONS,
                         pixels_per_cell=PIX_PER_CELL, cells_per_block=CELLS_PER_BLOCK,
                         transform_sqrt=True, visualize=True)
        hog_img = exposure.rescale_intensity(hog_img, in_range=(0, 10))
        axes[row, col+1].imshow(hog_img, cmap='hot')
        axes[row, col+1].set_title(f'{label}\nHOG – {ch_names[col]}', fontsize=10)
        axes[row, col+1].axis('off')

plt.tight_layout()
plt.savefig('02_hog_visualization.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Extract Features from ALL Images ────────────────────────────────────────
print('⚙️  Extracting HOG features — Cars...')
car_features = [extract_hog_features(p) for p in tqdm(car_paths)]
car_features = [f for f in car_features if f is not None]

print('⚙️  Extracting HOG features — Non-Cars...')
non_car_features = [extract_hog_features(p) for p in tqdm(non_car_paths)]
non_car_features = [f for f in non_car_features if f is not None]

print(f'\n✅ Car features     : {len(car_features)}')
print(f'✅ Non-car features : {len(non_car_features)}')

## 🤖 Step 3: Train SVM Classifier
بنستخدم **LinearSVC** — أسرع وأقوى لـ HOG features.

In [ ]:
# ─── Build Dataset ───────────────────────────────────────────────────────────
X = np.array(car_features + non_car_features, dtype=np.float32)
y = np.array([1]*len(car_features) + [0]*len(non_car_features))  # 1=car, 0=non-car

# 80/20 split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling — مهم جداً للـ SVM
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'📊 Train: {X_train.shape[0]} samples')
print(f'📊 Test : {X_test.shape[0]} samples')
print(f'⚖️  Train balance: {np.sum(y_train==1)} cars | {np.sum(y_train==0)} non-cars')

In [ ]:
# ─── Train ───────────────────────────────────────────────────────────────────
print('🏋️  Training LinearSVC...')
svm = LinearSVC(C=0.01, max_iter=5000, random_state=42)
svm.fit(X_train_sc, y_train)
print('✅ Training complete!')

## 📊 Step 4: Evaluate Classifier

In [ ]:
# ─── Metrics ─────────────────────────────────────────────────────────────────
y_pred = svm.predict(X_test_sc)
acc    = accuracy_score(y_test, y_pred)

print(f'🎯 Test Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
print()
print(classification_report(y_test, y_pred, target_names=['Non-Car', 'Car']))

In [ ]:
# ─── Confusion Matrix ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Non-Car', 'Car'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix  (Accuracy: {acc*100:.2f}%)', fontweight='bold')

# Class distribution
counts = [len(car_features), len(non_car_features)]
colors = ['#4CAF50', '#F44336']
bars   = axes[1].bar(['Car (Positive)', 'Non-Car (Negative)'], counts, color=colors, edgecolor='black', width=0.5)
for bar, count in zip(bars, counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 str(count), ha='center', va='bottom', fontsize=13, fontweight='bold')
axes[1].set_title('Dataset Class Distribution', fontweight='bold')
axes[1].set_ylabel('Number of Images')
axes[1].set_ylim(0, max(counts) * 1.15)

plt.tight_layout()
plt.savefig('03_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

## 🔍 Step 5: Sliding Window Detection

**الفكرة:** بنمشي على الصورة الكاملة بـ window صغيرة (64×64)، في كل موضع بنعمل:
1. Extract HOG features من الـ window
2. SVM يصنف: car أم non-car؟
3. لو car → نحفظ الـ bounding box

بعدين بنطبق **Non-Maximum Suppression (NMS)** عشان ندمج الـ overlapping boxes.

In [ ]:
# ─── Sliding Window + NMS Functions ──────────────────────────────────────────

def sliding_window_detect(image_bgr, win_size=(64,64), step=16, scales=[1.0, 1.5, 2.0]):
    """
    Multi-scale sliding window detection.
    Returns list of (x1, y1, x2, y2, score) boxes where car was detected.
    """
    h_orig, w_orig = image_bgr.shape[:2]
    detections = []

    for scale in scales:
        # Resize image for this scale
        new_w = int(w_orig / scale)
        new_h = int(h_orig / scale)
        if new_w < win_size[0] or new_h < win_size[1]:
            continue
        img_scaled = cv2.resize(image_bgr, (new_w, new_h))

        # Slide window
        for y in range(0, new_h - win_size[1] + 1, step):
            for x in range(0, new_w - win_size[0] + 1, step):
                patch = img_scaled[y:y+win_size[1], x:x+win_size[0]]

                # Extract HOG
                img_ycrcb = cv2.cvtColor(patch, cv2.COLOR_BGR2YCrCb)
                features  = []
                for ch in range(3):
                    feat = hog(img_ycrcb[:,:,ch], orientations=ORIENTATIONS,
                               pixels_per_cell=PIX_PER_CELL,
                               cells_per_block=CELLS_PER_BLOCK,
                               transform_sqrt=True, feature_vector=True)
                    features.append(feat)
                feat_vec = np.concatenate(features).reshape(1, -1)
                feat_vec = scaler.transform(feat_vec)

                # Classify
                score = svm.decision_function(feat_vec)[0]
                if svm.predict(feat_vec)[0] == 1:  # car detected
                    # Map coordinates back to original scale
                    x1 = int(x * scale);          y1 = int(y * scale)
                    x2 = int((x + win_size[0]) * scale); y2 = int((y + win_size[1]) * scale)
                    detections.append((x1, y1, x2, y2, score))

    return detections


def non_max_suppression(boxes, overlap_thresh=0.4):
    """
    NMS: بنحذف الـ overlapping boxes ونفضل الأقوى.
    Input:  list of (x1, y1, x2, y2, score)
    Output: list of (x1, y1, x2, y2)
    """
    if len(boxes) == 0:
        return []

    boxes_arr = np.array(boxes)
    x1 = boxes_arr[:, 0].astype(float)
    y1 = boxes_arr[:, 1].astype(float)
    x2 = boxes_arr[:, 2].astype(float)
    y2 = boxes_arr[:, 3].astype(float)
    scores = boxes_arr[:, 4]

    areas = (x2 - x1 + 1) * (y2 - y1 + 1)
    order = scores.argsort()[::-1]  # sort by score descending

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w = np.maximum(0, xx2 - xx1 + 1)
        h = np.maximum(0, yy2 - yy1 + 1)
        inter = w * h
        iou   = inter / (areas[i] + areas[order[1:]] - inter)

        inds  = np.where(iou <= overlap_thresh)[0]
        order = order[inds + 1]

    return [(int(x1[i]), int(y1[i]), int(x2[i]), int(y2[i])) for i in keep]


print('✅ Sliding window & NMS functions defined!')

## 🖼️ Step 6: Run Detection on Test Images
هنجرب الـ detector على صور من الداتاسيت نفسه (صور فيها عربيات)

In [ ]:
# ─── Run on sample car images ─────────────────────────────────────────────────
# بنبني صورة اختبار أكبر بـ paste عدة patches جنب بعض
# (لأن الداتاسيت صور 64x64، هنعمل test image مركبة)

def build_test_image(car_paths, non_car_paths, rows=3, cols=6):
    """Build a test scene by tiling car + non-car patches"""
    patches_list = []
    # 50% cars, 50% non-cars randomly placed
    import random
    random.seed(7)
    sources = random.choices(car_paths[:50], k=rows*cols//2) + \
              random.choices(non_car_paths[:50], k=rows*cols - rows*cols//2)
    random.shuffle(sources)

    grid_imgs = []
    for p in sources:
        img = cv2.resize(cv2.imread(p), (64, 64))
        grid_imgs.append(img)

    row_imgs = []
    for r in range(rows):
        row_imgs.append(np.hstack(grid_imgs[r*cols:(r+1)*cols]))
    return np.vstack(row_imgs), sources

test_image, sources = build_test_image(car_paths, non_car_paths, rows=3, cols=6)
print(f'Test image size: {test_image.shape}')

# Run sliding window detection
print('🔍 Running sliding window detection...')
raw_boxes = sliding_window_detect(test_image, win_size=(64,64), step=32,
                                   scales=[1.0])  # scale=1.0 since patches are already 64x64
nms_boxes = non_max_suppression(raw_boxes, overlap_thresh=0.3)

print(f'📦 Raw detections : {len(raw_boxes)}')
print(f'📦 After NMS      : {len(nms_boxes)}')

In [ ]:
# ─── Draw Bounding Boxes & Display ───────────────────────────────────────────
def draw_boxes(img_bgr, boxes, color=(0, 255, 0), thickness=2):
    """Draw bounding boxes on image (returns RGB copy)"""
    img_copy = img_bgr.copy()
    for (x1, y1, x2, y2) in boxes:
        cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, thickness)
        cv2.putText(img_copy, 'Car', (x1, max(y1-5, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Sliding Window Detection Results', fontsize=15, fontweight='bold')

# Original
axes[0].imshow(cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Test Image', fontsize=12)
axes[0].axis('off')

# All raw detections (before NMS)
raw_img = draw_boxes(test_image, [(b[0],b[1],b[2],b[3]) for b in raw_boxes],
                     color=(0, 165, 255))
axes[1].imshow(raw_img)
axes[1].set_title(f'Raw Detections ({len(raw_boxes)} boxes) — Before NMS', fontsize=12)
axes[1].axis('off')

# After NMS
nms_img = draw_boxes(test_image, nms_boxes, color=(0, 255, 0))
axes[2].imshow(nms_img)
axes[2].set_title(f'After NMS ({len(nms_boxes)} boxes) ✅', fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.savefig('04_sliding_window_detection.png', dpi=120, bbox_inches='tight')
plt.show()

## 🚗 Step 7: Detect on a REAL Road Image
بنجرب على صورة حقيقية من الإنترنت (أو اي صورة تحبيها)

In [ ]:
import urllib.request

# Download a real road image for testing
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/5/5c/US_highway_30_West_Junction_US_30S.jpg/640px-US_highway_30_West_Junction_US_30S.jpg'
urllib.request.urlretrieve(url, 'test_road.jpg')

real_img = cv2.imread('test_road.jpg')
print(f'Real image size: {real_img.shape}')

# Multi-scale detection
print('🔍 Running multi-scale detection on real image...')
raw_real = sliding_window_detect(real_img, win_size=(64,64), step=16,
                                  scales=[1.0, 1.5, 2.0, 3.0])
nms_real = non_max_suppression(raw_real, overlap_thresh=0.3)

print(f'Raw: {len(raw_real)} | After NMS: {len(nms_real)}')

# Show result
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].imshow(cv2.cvtColor(real_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Road Image', fontsize=13)
axes[0].axis('off')

result_img = draw_boxes(real_img, nms_real)
axes[1].imshow(result_img)
axes[1].set_title(f'Detected Cars: {len(nms_real)} bounding boxes', fontsize=13)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('05_real_image_detection.png', dpi=120, bbox_inches='tight')
plt.show()

## 📝 Step 8: Full Evaluation Summary

In [ ]:
# ─── Final Summary ────────────────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print('=' * 50)
print('        📊 EVALUATION SUMMARY')
print('=' * 50)
print(f'  Dataset        : Car and Non-Car (Kaggle)')
print(f'  Total samples  : {len(X)}')
print(f'  Cars           : {len(car_features)}')
print(f'  Non-Cars       : {len(non_car_features)}')
print(f'  Train / Test   : {len(X_train)} / {len(X_test)}')
print(f'  Feature size   : {feat_len}')
print('-' * 50)
print(f'  Classifier     : LinearSVC  (C=0.01)')
print(f'  Color Space    : YCrCb (3 channels)')
print(f'  HOG cells/pix  : {PIX_PER_CELL}')
print(f'  HOG cells/block: {CELLS_PER_BLOCK}')
print(f'  Orientations   : {ORIENTATIONS}')
print('-' * 50)
print(f'  ✅ Accuracy    : {acc*100:.2f}%')
print(f'  ✅ Precision   : {precision*100:.2f}%')
print(f'  ✅ Recall      : {recall*100:.2f}%')
print(f'  ✅ F1-Score    : {f1*100:.2f}%')
print('-' * 50)
print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')
print('=' * 50)

In [ ]:
# ─── Metrics Bar Chart ───────────────────────────────────────────────────────
metrics = {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1-Score': f1}
colors  = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(metrics.keys(), [v*100 for v in metrics.values()],
              color=colors, edgecolor='black', width=0.5)
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val*100:.2f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_ylim(0, 115)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Classifier Evaluation Metrics — HOG + LinearSVC', fontsize=14, fontweight='bold')
ax.axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% threshold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('06_metrics_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n🎉 Pipeline complete! All results saved.')